In [1]:
import os
import tarfile
import random
import re
import numpy as np
import pandas as pd
import torch
import librosa
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader, Subset
from jiwer import wer
from tqdm.auto import tqdm
import whisper

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 50
    lr = 3e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"

    L = 5
    H = 64
    U = 4
    target_samples = 16384*2

In [4]:
def init_weights(m):
    if isinstance(m, nn.Conv1d) or isinstance(m, nn.ConvTranspose1d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def cumulative_std(x, eps=1e-8):
    cum_sq_sum = torch.cumsum(x ** 2, dim=-1)
    t = torch.arange(1, x.shape[-1] + 1, device=x.device, dtype=x.dtype).view(1, 1, -1)
    std = torch.sqrt(cum_sq_sum / t + eps)
    return std

class EncoderLayer(nn.Module):
    def __init__(self, in_channels, out_channels, K=8, S=4):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, K, S)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels * 2, 1)
        self.glu = nn.GLU(dim=1) 
        self.pad_left = K - S

    def forward(self, x):
        x = F.pad(x, (self.pad_left, 0))
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.glu(x)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, in_channels, out_channels, K=8, S=4, is_last=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, in_channels * 2, 1)
        self.glu = nn.GLU(dim=1)
        self.conv_tr = nn.ConvTranspose1d(in_channels, out_channels, K, S)
        self.is_last = is_last
        self.crop_right = K - S

        if not is_last:
            self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.glu(x)
        x = self.conv_tr(x)
        
        if self.crop_right > 0:
            x = x[..., :-self.crop_right]
            
        if not self.is_last:
            x = self.relu(x)
        return x

class CausalDEMUCS(nn.Module):
    def __init__(self, L=5, H=64, K=8, S=4, U=4, sample_rate=16000, lookahead_ms=3):
        super().__init__()
        self.U = U
        self.lookahead_samples = int(sample_rate * lookahead_ms / 1000) # 
        
        self.resample_up = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=sample_rate * U)
        self.resample_down = torchaudio.transforms.Resample(orig_freq=sample_rate * U, new_freq=sample_rate)

        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        in_channels = 1
        for i in range(1, L + 1):
            out_channels = (2 ** (i - 1)) * H
            self.encoder.append(EncoderLayer(in_channels, out_channels, K, S))
            in_channels = out_channels

        self.lstm = nn.LSTM(in_channels, in_channels, num_layers=2, batch_first=True)

        for i in range(L, 0, -1):
            in_channels = (2 ** (i - 1)) * H
            out_channels = (2 ** (i - 2)) * H if i > 1 else 1
            self.decoder.append(DecoderLayer(in_channels, out_channels, K, S, is_last=(i == 1)))

        self.apply(init_weights)

    def forward(self, x):
        std = cumulative_std(x)
        x = x / std

        if self.lookahead_samples > 0:
            x = F.pad(x, (0, self.lookahead_samples))
        x = self.resample_up(x)

        total_stride = 4 ** len(self.encoder) # S=4, L=5
        pad_len = (total_stride - (x.shape[-1] % total_stride)) % total_stride
        if pad_len > 0:
            x = F.pad(x, (0, pad_len))
        
        skip_connections = []
        for enc in self.encoder:
            x = enc(x)
            skip_connections.append(x)

        x_perm = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x_perm)
        x = lstm_out.permute(0, 2, 1) + x

        for i, dec in enumerate(self.decoder):
            skip = skip_connections[-(i + 1)]
            
            if skip.shape[-1] != x.shape[-1]:
                min_len = min(skip.shape[-1], x.shape[-1])
                x = x[..., :min_len] + skip[..., :min_len]
            else:
                x = x + skip
                
            x = dec(x)

        x = self.resample_down(x)

        if self.lookahead_samples > 0:
            x = x[..., self.lookahead_samples:]

        min_len = min(x.shape[-1], std.shape[-1])
        x = x[..., :min_len] * std[..., :min_len]

        return x

class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, 
                 fft_sizes=[512, 1024, 2048], 
                 hop_sizes=[50, 120, 240], 
                 win_lengths=[240, 600, 1200]):
        super().__init__()
        self.stft_params = list(zip(fft_sizes, hop_sizes, win_lengths))

    def stft(self, x, n_fft, hop_length, win_length):
        x = x.squeeze(1)
        window = torch.hann_window(win_length).to(x.device)
        stft_out = torch.stft(x, n_fft=n_fft, hop_length=hop_length, 
                              win_length=win_length, window=window, 
                              return_complex=True)
        return torch.abs(stft_out)

    def stft_loss(self, y, y_hat, n_fft, hop_length, win_length):
        stft_y = self.stft(y, n_fft, hop_length, win_length)
        stft_y_hat = self.stft(y_hat, n_fft, hop_length, win_length)

        sc_loss = torch.norm(stft_y - stft_y_hat, p='fro') / torch.norm(stft_y, p='fro')

        mag_loss = F.l1_loss(torch.log(stft_y + 1e-7), torch.log(stft_y_hat + 1e-7))

        return sc_loss + mag_loss

    def forward(self, y, y_hat):
        l1_loss = F.l1_loss(y, y_hat) 
        stft_loss_total = 0.0
        
        for n_fft, hop_length, win_length in self.stft_params:
            stft_loss_total += self.stft_loss(y, y_hat, n_fft, hop_length, win_length)
            
        return l1_loss + stft_loss_total

In [5]:
class WaveformDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec 
        
        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)

    def __len__(self): 
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        wav = torch.from_numpy(wav_np)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
        
        if sr != Config.sr: 
            wav = T.Resample(sr, Config.sr)(wav)
        if wav.shape[0] > 1: 
            wav = wav.mean(dim=0, keepdim=True)

        max_samples = int(self.max_len_sec * Config.sr)
        
        if self.is_train:
            if wav.shape[-1] > max_samples:
                start = random.randint(0, wav.shape[-1] - max_samples)
                wav = wav[:, start:start + max_samples]
            else:
                wav = F.pad(wav, (0, max_samples - wav.shape[-1]))
        else:
            if wav.shape[-1] > max_samples:
                wav = wav[:, :max_samples]
            else:
                wav = F.pad(wav, (0, max_samples - wav.shape[-1]))

        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean
        
        return noisy, clean, self.ref_dict[os.path.basename(file_path)]

In [6]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [7]:
if not os.path.exists(Config.extract_dir):
        os.makedirs(Config.extract_dir, exist_ok=True)
        with tarfile.open(Config.tar_path, "r") as tar: 
            tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [8]:
model = CausalDEMUCS(L=Config.L, H=Config.H, U=Config.U).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
criterion = MultiResolutionSTFTLoss()

dataset = WaveformDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size],generator = generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)

In [ ]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        optimizer.zero_grad()

        pred = model(noisy)
        
        loss = criterion(clean, pred)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
    
torch.save(model.state_dict(), f"../models/demucs_weights.pth")

Epoch 1:   0%|          | 4/5954 [00:10<4:26:51,  2.69s/it, Loss=15.6673]


KeyboardInterrupt: 

In [10]:
model.load_state_dict(torch.load('../models/demucs_weights.pth'))

FileNotFoundError: [Errno 2] No such file or directory: '../models/demucs_weights.pth'

In [ ]:
seed_everything(42)

In [ ]:
def evaluate_demucs(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):
            
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav, force_type=n_type, file_seed=idx).to(device)

                noisy_input = noisy_wav.unsqueeze(0) 
                
                denoised_wav = model(noisy_input)

                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                
                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_demucs(model, Config.device, val_ds, limit=20)

WER Eval:   0%|          | 0/20 [00:00<?, ?it/s]


Noise Type | WER Noisy  | WER Denoised
babble     | 0.5376     | 0.6779    
rir        | 1.0000     | 1.0000    
white      | 0.5670     | 0.7446    


In [ ]:
from jiwer import process_words
import numpy as np
import torch
import whisper
from tqdm.auto import tqdm

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            
            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav, force_type=n_type, file_seed=idx).to(device)
                noisy_tensor = noisy_wav.unsqueeze(0)
                
                denoised_tensor = model(noisy_tensor)
                
                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()
                
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
                
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
                
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
                
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
        
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
        
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
        
        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")
        
seed_everything(42)
evaluate_and_listen_components(model, Config.device, val_ds, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
-----------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.6779 (0.2487/0.4075/0.0217) | -0.1403 
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0000 (0.2618/0.7382/0.0000) | 0.0000  
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.7446 (0.3269/0.3832/0.0345) | -0.1776 
